<a href="https://www.kaggle.com/code/avikdas567/aimo3-qwen-math-llm-with-dual-pass-reasoning-flow?scriptVersionId=307202779" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# AIMO 3: Qwen Math LLM with Dual-Pass Reasoning Flow

This notebook presents a **lightweight LLM-based reasoning pipeline** for solving olympiad-level mathematical problems from the *AI Mathematical Olympiad - Progress Prize 3* competition. The approach is built around a **dual-pass prompting strategy** using a compact math-specialized language model. The goal is to balance reasoning quality with efficient inference.

---

## Methodology

### Model

* **Qwen2.5-Math-1.5B-Instruct**
* Loaded locally (no internet dependency)
* FP16 inference on GPU

### Dual-Pass Reasoning

Each problem is solved using two complementary passes:

#### 1. **Solver Pass**

* Generates a quick candidate answer
* Uses both low and high temperature sampling for diversity

#### 2. **Checker Pass**

* Generates a secondary candidate using a more conservative prompt

### Answer Selection Strategy

The final answer is chosen using a simple but effective hierarchy:

1. If solver outputs agree → return immediately
2. Otherwise → include checker output
3. Select using pairwise agreement with checker-prioritized fallback

### Answer Extraction

Robust parsing handles multiple output formats:

* `\\boxed{}` expressions
* Explicit `Answer =` patterns
* Final numeric tokens

All outputs are normalized to integers in `[0, 99999]`.

---

In [1]:
import os
import re
import torch
import random

import pandas as pd
import polars as pl
from transformers import AutoTokenizer, AutoModelForCausalLM

import kaggle_evaluation.aimo_3_inference_server

# CONFIG

MODEL_PATH = "/kaggle/input/models/alekseizabirnik/qwen2.5-math-1.5b-instruct/transformers/default/1"
MAX_NEW_TOKENS = 80

# MODEL

class Model:
    def __init__(self):
        self.model = None
        self.tokenizer = None

    def load(self):
        print("Loading model...")
        self.tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
        self.model = AutoModelForCausalLM.from_pretrained(
            MODEL_PATH,
            torch_dtype=torch.float16,
            device_map="auto"
        )
        self.model.eval()

    def generate(self, prompt, temp):
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

        with torch.no_grad():
            out = self.model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=temp,
                top_p=0.9,
                repetition_penalty=1.1,
                pad_token_id=self.tokenizer.eos_token_id
            )

        return self.tokenizer.decode(out[0], skip_special_tokens=True)


model = Model()

# PROMPTS

def solver_prompt(problem):
    return f"""
Solve quickly. Return only final integer (0–99999).

Problem:
{problem}

Answer:
"""

def checker_prompt(problem):
    return f"""
Solve carefully and verify the answer.

Return only the final integer.

Problem:
{problem}

Final Answer:
"""

# EXTRACTION

def extract(text):
    text = text.lower()

    patterns = [
        r'\\boxed{(\d+)}',
        r'answer\s*[:=]\s*(\d+)',
        r'final answer\s*[:=]?\s*(\d+)'
    ]

    for p in patterns:
        m = re.search(p, text)
        if m:
            return int(m.group(1)) % 100000

    nums = re.findall(r'\d+', text)
    if nums:
        return int(nums[-1]) % 100000

    return None

# SOLVER

def solve(problem):
    if model.model is None:
        model.load()

    out1 = model.generate(solver_prompt(problem), 0.2)
    out2 = model.generate(solver_prompt(problem), 0.7)

    ans1 = extract(out1)
    ans2 = extract(out2)

    if ans1 is not None and ans1 == ans2:
        return ans1

    out3 = model.generate(checker_prompt(problem), 0.3)
    ans3 = extract(out3)

    candidates = [ans1, ans2, ans3]
    candidates = [a for a in candidates if a is not None]

    if not candidates:
        return random.randint(0, 99999)

    for a in candidates:
        if candidates.count(a) > 1:
            return a

    if ans3 is not None:
        return ans3

    return candidates[0]

# PREDICT

def predict(id_: pl.Series, problem: pl.Series):
    id_ = id_.item(0)
    problem_text = problem.item(0)

    answer = solve(problem_text)

    return pl.DataFrame({
        "id": id_,
        "answer": int(answer)
    })

# SERVER

inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(
    predict
)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(
        ('/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/test.csv',)
    )

Loading model...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]